# Part 1 - 실습 1: Amazon Bedrock 기초 & Foundation Model 탐색
**소요시간: 50분** | 난이도: ⭐⭐

## 학습 목표
1. Amazon Bedrock 클라이언트를 초기화하고 사용 가능한 Foundation Model 목록을 조회합니다.
2. `invoke_model` API로 Claude에게 첫 번째 프롬프트를 전송합니다.
3. `converse` API(통합 인터페이스)로 멀티턴 대화를 구현합니다.

## Amazon Bedrock 아키텍처
```
사용자 코드
   │
   ├─ bedrock          (모델 목록 조회)
   ├─ bedrock-runtime  (텍스트 생성, 스트리밍)
   └─ bedrock-agent-runtime  (RAG / Knowledge Base)
```

## 주요 모델 ID (2025 기준)
```
anthropic.claude-3-5-sonnet-20241022-v2:0  ← 최신 고성능
anthropic.claude-3-haiku-20240307-v1:0    ← 빠르고 경제적
amazon.titan-text-premier-v1:0            ← AWS 자체 모델
meta.llama3-8b-instruct-v1:0              ← 오픈소스 계열
```


---
## 🏢 기업 시나리오 — 생성형 AI 도입 TF

당신은 회사의 **생성형 AI 도입 TF 담당자**입니다.
"GPT 같은 걸 우리 업무에도 써보자"는 요청을 받았습니다. 모델을 직접 학습시키지 않고 **API 호출만으로** 시작합니다.

이번 실습에서는 다음을 수행합니다.
1. **모델 조사** → 어떤 Foundation Model이 있는지 목록 조회 (list_foundation_models)
2. **첫 호출** → Claude에게 프롬프트를 보내 응답 받기 (invoke_model)
3. **멀티턴 대화** → 챗봇의 토대가 되는 대화 구현 (converse)

> 💡 모델 선택은 비용·속도·품질의 trade-off입니다. 보통 저렴·빠른 모델(Haiku)로 PoC를 시작하고, 품질이 부족하면 상위 모델로 올립니다.


In [2]:
# ✅ [제공 코드] 환경 초기화
import boto3, json
import pandas as pd

# Bedrock 클라이언트 (모델 목록 조회용)
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Bedrock Runtime 클라이언트 (텍스트 생성용)
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

print('✅ Bedrock 클라이언트 생성 완료')
print(f'  bedrock           : {bedrock.meta.service_model.service_name}')
print(f'  bedrock-runtime   : {bedrock_runtime.meta.service_model.service_name}')


✅ Bedrock 클라이언트 생성 완료
  bedrock           : bedrock
  bedrock-runtime   : bedrock-runtime


## ✏️ TODO 1: Foundation Model 목록 조회

Bedrock에서 사용 가능한 Foundation Model 목록을 조회하고 텍스트 생성 모델만 필터링하세요.


In [3]:
# ✏️ TODO 1: list_foundation_models API로 사용 가능한 모델을 조회하세요
response = bedrock.list_foundation_models(
    byOutputModality='TEXT'    # ← 'TEXT'
)

models = response['modelSummaries']   # ← 'modelSummaries'
print(f'텍스트 생성 모델 수: {len(models)}개\n')

rows = []
for m in models:
    rows.append({
        '모델ID'    : m['modelId'],                 # ← 'modelId'
        '제공사'    : m['providerName'],                 # ← 'providerName'
        '모델명'    : m['modelName'],
    })

df = pd.DataFrame(rows)
print(df[df['모델ID'].str.contains('claude|titan|llama')].to_string(index=False))


텍스트 생성 모델 수: 94개

                                        모델ID       제공사                           모델명
     anthropic.claude-sonnet-4-20250514-v1:0 Anthropic               Claude Sonnet 4
    anthropic.claude-haiku-4-5-20251001-v1:0 Anthropic              Claude Haiku 4.5
                    anthropic.claude-fable-5 Anthropic                Claude Fable 5
                 anthropic.claude-sonnet-4-6 Anthropic             Claude Sonnet 4.6
                anthropic.claude-opus-4-6-v1 Anthropic               Claude Opus 4.6
                   anthropic.claude-opus-4-8 Anthropic               Claude Opus 4.8
                   anthropic.claude-opus-4-7 Anthropic               Claude Opus 4.7
   anthropic.claude-sonnet-4-5-20250929-v1:0 Anthropic             Claude Sonnet 4.5
                   anthropic.claude-sonnet-5 Anthropic               Claude Sonnet 5
     anthropic.claude-opus-4-1-20250805-v1:0 Anthropic               Claude Opus 4.1
     anthropic.claude-opus-4-5-20251101-v1:0 An

## ✏️ TODO 2: invoke_model — 첫 번째 텍스트 생성

Claude에게 프롬프트를 전송하고 응답을 파싱하세요. `invoke_model`은 모델별 고유 페이로드 구조를 사용합니다.

```python
# Claude 3 페이로드 구조
{
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [{'role': 'user', 'content': '프롬프트 내용'}]
}
```


In [4]:
# ✏️ TODO 2: invoke_model API로 Claude에게 질문을 전송하세요
# us-east-1이면 프리픽스는 us. 입니다.
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,           # ← 1024
    'messages': [
        {
            'role': 'user',         # ← 'user'
            'content': 'AWS Bedrock을 한 문장으로 설명해줘.'       # ← 'AWS Bedrock을 한 문장으로 설명해줘.'
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,                 # ← MODEL_ID
    body=json.dumps(payload),        # ← payload
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['content'][0]['text']   # ← 'content', 'text'
print('Claude 응답:')
print(answer)


Claude 응답:
AWS Bedrock은 Amazon, Anthropic, Cohere 등 다양한 기업의 대규모 언어 모델(LLM)을 API를 통해 쉽게 사용할 수 있게 해주는 AWS의 완전관리형 생성 AI 서비스입니다.


In [5]:
# ✏️ TODO 2-2: invoke_model API로 Nova에게 질문을 전송하세요
MODEL_ID = 'amazon.nova-micro-v1:0'

payload = {
    'schemaVersion': 'messages-v1',    # Nova 전용 필드
    'inferenceConfig': {'maxTokens': 1024},  # 힌트: 응답으로 생성할 최대 토큰 수 (정수)
    'messages': [
        {
            'role': 'user',             # 힌트: 대화에서 사람 쪽 역할을 나타내는 문자열
            'content': [{'text': 'AWS Bedrock을 한 문장으로 설명해줘.'}]  # 힌트: 모델에게 보낼 프롬프트 문자열을 입력하세요
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,                     # 힌트: 이 셀 상단에 정의된 모델 ID 변수를 참조하세요
    body=json.dumps(payload),             # 힌트: 위에서 정의한 페이로드 딕셔너리를 JSON 직렬화해 전달하세요
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['output']['message']['content'][0]['text']
print('Nova 응답:')
print(answer)

Nova 응답:
AWS Bedrock은 고객이 맞춤형 AI 모델을 개발하고 구축할 수 있도록 지원하는 클라우드 기반의 서비스입니다.


In [12]:
# ✅ [제공 코드] Temperature & Max Tokens 파라미터 실험
# ⚠️ 이 셀은 그대로 실행하면 ValidationException이 발생합니다. 왜 그런지 생각해 보세요!
#    힌트: 직전 셀(TODO 2-2)에서 MODEL_ID가 'amazon.nova-micro-v1:0'으로 재할당되었는데,
#          아래 payload는 Claude 전용 형식(anthropic_version, temperature, ...)입니다.
#          Nova는 이 키들을 모르기 때문에 "extraneous key is not permitted" 에러를 냅니다.
#    해결: 아래 줄의 주석을 해제해 MODEL_ID를 Claude로 되돌린 뒤 다시 실행하세요.
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

def call_claude(prompt, temperature=0.7, max_tokens=512):
    payload = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'temperature': temperature,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    resp = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(payload), contentType='application/json'
    )
    return json.loads(resp['body'].read())['content'][0]['text']

prompt = '파이썬의 장점을 세 가지만 알려줘.'
for temp in [0.1, 0.7, 1.0]: # 1.2 왜 안되지? 1.1도 안된다. 아뭐야 바꾸고시작하래
    print(f'\n--- temperature={temp} ---')
    print(call_claude(prompt, temperature=temp))



--- temperature=0.1 ---


파이썬의 주요 장점 3가지를 소개해드리겠습니다.

## 1. **간단하고 읽기 쉬운 문법**
- 영어와 유사한 직관적인 문법으로 코드 작성이 쉽습니다
- 들여쓰기로 코드 블록을 구분해 가독성이 뛰어납니다
- 초보자도 빠르게 학습할 수 있습니다

## 2. **풍부한 라이브러리 생태계**
- 데이터 분석(pandas, numpy), 웹 개발(Django, Flask), AI/ML(tensorflow, scikit-learn) 등 다양한 분야의 라이브러리가 풍부합니다
- pip를 통해 쉽게 외부 패키지를 설치하고 관리할 수 있습니다

## 3. **높은 생산성과 개발 속도**
- 적은 코드로 많은 기능을 구현할 수 있습니다
- 프로토타이핑과 빠른 개발에 매우 적합합니다
- 디버깅과 테스트가 용이합니다

이러한 장점들로 인해 파이썬은 웹 개발, 데이터 과학, 인공지능, 자동화 등 다양한 분야에서 널리 사용되고 있습니다.

--- temperature=0.7 ---


파이썬의 주요 장점 3가지를 소개해드리겠습니다.

## 1. **간단하고 읽기 쉬운 문법**
- 영어와 유사한 직관적인 문법 구조
- 들여쓰기로 코드 블록을 구분하여 가독성이 뛰어남
- 초보자도 쉽게 배우고 이해할 수 있음

## 2. **풍부한 라이브러리 생태계**
- 표준 라이브러리가 매우 다양하고 강력함
- PyPI(Python Package Index)를 통해 수십만 개의 서드파티 패키지 제공
- 웹 개발, 데이터 분석, AI/ML, 자동화 등 다양한 분야의 라이브러리 활용 가능

## 3. **다양한 분야에서의 활용성**
- 웹 개발 (Django, Flask)
- 데이터 사이언스 (pandas, numpy, matplotlib)
- 인공지능/머신러닝 (tensorflow, pytorch, scikit-learn)
- 자동화/스크립팅, 게임 개발 등 멀티 플랫폼 지원

이러한 장점들로 인해 파이썬은 전 세계적으로 가장 인기 있는 프로그래밍 언어 중 하나가 되었습니다.

--- temperature=1.0 ---


파이썬의 주요 장점 3가지를 소개해드리겠습니다.

## 1. **간단하고 읽기 쉬운 문법**
- 영어와 유사한 직관적인 문법으로 초보자도 쉽게 학습할 수 있습니다
- 들여쓰기를 통한 코드 구조화로 가독성이 뛰어납니다

```python
# 파이썬
if age >= 18:
    print("성인입니다")
```

## 2. **풍부한 라이브러리 생태계**
- 데이터 분석(pandas, numpy), 웹 개발(Django, Flask), AI/ML(tensorflow, scikit-learn) 등 다양한 분야의 강력한 라이브러리들이 풍부합니다
- pip를 통해 쉽게 설치하고 활용할 수 있습니다

## 3. **높은 생산성과 개발 속도**
- 적은 코드로 많은 기능을 구현할 수 있어 개발 시간이 단축됩니다
- 프로토타이핑부터 실제 서비스까지 빠르게 개발 가능합니다

이러한 장점들로 인해 파이썬은 개발 입문자부터 전문가까지 폭넓게 사용되고 있습니다.


## ✏️ TODO 3: converse API — 통합 멀티턴 대화

`converse`는 모델별 페이로드 차이를 추상화한 통합 API입니다. 대화 히스토리를 누적하여 멀티턴을 구현하세요.

```python
# converse 구조
bedrock_runtime.converse(
    modelId='...',
    messages=[{'role': 'user', 'content': [{'text': '...'}]}],
    inferenceConfig={'maxTokens': 512, 'temperature': 0.7}
)
```


In [13]:
# ✏️ TODO 3: converse API로 멀티턴 대화를 구현하세요
# 💡 converse API는 모델별 payload 형식이 필요 없는 통합 인터페이스입니다.
#    invoke_model과 달리 Claude든 Nova든 아래 코드는 그대로 두고 MODEL_ID만 바꾸면 동작합니다.
#    두 모델로 각각 실행해 보세요.
MODEL_ID = 'amazon.nova-micro-v1:0'
# MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

conversation_history = []

def chat(user_message, system_prompt=None):
    """대화 히스토리를 유지하며 converse API를 호출합니다"""
    conversation_history.append({
        'role': 'user',                           # ← 'user'
        'content': [{'text': user_message}]             # ← user_message
    })

    kwargs = {
        'modelId': MODEL_ID,
        'messages': conversation_history,          # ← conversation_history
        'inferenceConfig': {'maxTokens': 512, 'temperature': 0.7}
    }
    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]

    response = bedrock_runtime.converse(**kwargs)

    assistant_msg = response['output']['message']['content'][0]['text']  # ← 'message', 'text'
    conversation_history.append({
        'role': 'assistant',
        'content': [{'text': assistant_msg}]
    })
    return assistant_msg

# 멀티턴 대화 테스트
SYSTEM = '당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요.'

print('사용자:', '안녕하세요! 클라우드 컴퓨팅이 뭔가요?')
print('AI:', chat('안녕하세요! 클라우드 컴퓨팅이 뭔가요?', SYSTEM))
print()
print('사용자:', 'AWS와 Azure 중 어떤 게 더 좋아요?')
print('AI:', chat('AWS와 Azure 중 어떤 게 더 좋아요?'))
print()
print(f'대화 히스토리 길이: {len(conversation_history)}개 메시지')


사용자: 안녕하세요! 클라우드 컴퓨팅이 뭔가요?


AI: 안녕하세요! 클라우드 컴퓨팅은 인터넷을 통해 다양한 컴퓨팅 자원을 원격으로 제공하는 기술입니다. 이러한 자원에는 서버, 스토리지, 네트워킹 장비, 소프트웨어, 데이터베이스 등이 포함됩니다. 사용자들은 이러한 자원을 필요에 따라 임시로 사용할 수 있으며, 지불 시스템은 종종 사용량에 따라 요금을 청구합니다.

클라우드 컴퓨팅의 주요 이점은 다음과 같습니다:

1. **비용 효율성**: 사전에 컴퓨팅 자원을 구매하거나 유지하지 않아도 되므로 초기 투자 비용을 줄일 수 있습니다.
2. **확장성**: 필요에 따라 자원을 쉽게 확장하거나 축소할 수 있습니다.
3. **유연성**: 다양한 지역에서 액세스할 수 있으므로 글로벌 비즈니스 운영에 유리합니다.
4. **업데이트 및 유지 관리**: 서비스 제공업체가 하드웨어와 소프트웨어를 관리하므로 업데이트 및 보안 관리 등의 책임이 사용자에게 덜 달아붙습니다.

��

사용자: AWS와 Azure 중 어떤 게 더 좋아요?


AI: AWS(Amazon Web Services)와 Azure는 모두 클라우드 컴퓨팅 서비스 중 가장 큰 두 플랫폼으로, 각각 다양한 기능과 특징을 제공합니다. 어느 하나를 선택하는 것은 여러 요소에 따라 달라질 수 있습니다. 여기에 몇 가지 고려 사항을 소개하겠습니다.

### AWS

**강점**

1. **시장 지배력**: AWS는 클라우드 컴퓨팅 시장에서 가장 큰 시장 점유율을 가지고 있습니다. 이는 그 안의 서비스와 도구가 더 많고 다양하다는 것을 의미합니다.
2. **서비스 다양성**: 가장 많은 서비스를 제공합니다. 예를 들어 머신러닝, 데이터 분석, 개발 도구 등을 다양하게 제공합니다.
3. **지역 다양성**: AWS는 전 세계 160개 이상의 지역에서 서비스를 제공합니다.

**약점**

1. **복잡성**: AWS의 많은 양의 서비스와 도구는 처음 사용자에게는 어렵게 느껴질 수 있습니다.
2. **비용**: 특히 초보자나 작은 회사의 경우 비용이 높아질 수 있습니다.

### Azure

**강점**

1. **마이크로소프트와의 통합**:

대화 히스토리 길이: 4개 메시지


---
## 🔗 실무로 연결하기

`사용자 입력` → `Lambda` → `Bedrock(converse)` → `응답` 의 흐름이 모든 생성형 AI 앱의 기본 골격입니다.

- **모델 선택 기준**: Haiku(빠름·저렴, 대량 처리) / Sonnet·Opus(고품질, 복잡 추론). 비용은 토큰 사용량 기반.
- PoC는 저렴한 모델로 시작 → 품질 검증 후 상위 모델 또는 프롬프트 개선.
- `converse` API는 모델이 바뀌어도 코드를 거의 그대로 재사용할 수 있어 실무에서 선호됩니다.


## 💡 심화 도전
1. `meta.llama3-8b-instruct-v1:0`을 `converse`로 호출하여 Claude와 같은 질문에 대한 응답을 비교해보세요.
2. Temperature를 0.01로 설정했을 때와 1.5로 설정했을 때 동일 프롬프트 결과가 어떻게 달라지는지 확인하세요.
3. `converse`의 `system` 파라미터로 다른 성격의 AI 페르소나를 설정해보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1
response = bedrock.list_foundation_models(byOutputModality='TEXT')
models = response['modelSummaries']
m['modelId'], m['providerName']

# TODO 2
payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [{'role': 'user', 'content': 'AWS Bedrock을 한 문장으로 설명해줘.'}]
}
response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(payload), ...)
answer = result['content'][0]['text']

# TODO 3
conversation_history.append({'role': 'user', 'content': [{'text': user_message}]})
kwargs = {'modelId': MODEL_ID, 'messages': conversation_history, ...}
assistant_msg = response['output']['message']['content'][0]['text']
```
